## Timeseries (Dask vs Ray)

This notebook is based on the [Working with Timeseries docs](https://docs.lsdb.io/en/latest/tutorials/pre_executed/timeseries.html) and compares Dask vs Ray scheduling.

In [1]:
import lsdb
lsdb.show_versions()


--------      SYSTEM INFO      --------
python        : 3.13.12
python-bits   : 64
OS            : Linux
OS-release    : 5.14.0-611.30.1.el9_7.x86_64
Version       : #1 SMP PREEMPT_DYNAMIC Fri Feb 13 17:04:55 UTC 2026
machine       : x86_64
processor     : x86_64
byteorder     : little
LC_ALL        : 
LANG          : C.UTF-8
--------   INSTALLED VERSIONS   --------
lsdb          : 0.9.0
hats          : 0.9.0
nested-pandas : 0.6.8
pandas        : 2.3.3
numpy         : 2.4.2
dask          : 2026.3.0
pyarrow       : 23.0.1
fsspec        : 2026.3.0


### Lazy construction of task graph

In [2]:
ztf_objects = lsdb.open_catalog(
    "https://data.lsdb.io/hats/ztf_dr22",
    search_filter=lsdb.ConeSearch(ra=254.5, dec=35.3, radius_arcsec=5*1800),
).nest_lists(
    list_columns=["hmjd", "mag", "magerr", "clrcoeff", "catflags"],
    name="lc",  # name to give the resulting nested column
)

In [3]:
filtered_catalog = ztf_objects.query("filterid == 2")
filtered_catalog = filtered_catalog.query("lc.catflags == 0")
# Using map_partitions to drop rows with NaN in lc.mag
filtered_catalog = filtered_catalog.map_partitions(lambda df: df.dropna(subset=["lc.mag"]))

In [4]:
import numpy as np

def median_mag(row):
    return np.median(row["lc.mag"])

catalog_w_features = filtered_catalog.map_rows(
    median_mag,  # our user-defined function
    columns=["lc.mag"],  # names of the column(s) to pass to the function
    output_names=["median_mag"],  # name(s) of the new column(s)
    meta={"median_mag": float},  # for Dask: describe the type of the output
    append_columns=True,
)

In [5]:
catalog_w_features = catalog_w_features.query("median_mag < 16")

In [6]:
from nested_pandas.utils import count_nested

def count_points(pts):
    # Asked to count `lc`, this will add a column called `n_lc`
    return count_nested(pts, "lc")

catalog_w_features = catalog_w_features.map_partitions(count_points)
catalog_w_features = catalog_w_features.query("n_lc >= 10")

In [7]:
from astropy.timeseries import LombScargle

def extract_period(row):
    time = row["lc.hmjd"]
    mag = row["lc.mag"]
    error = row["lc.magerr"]
    ls = LombScargle(time, mag, error)
    freq, power = ls.autopower()
    argmax = np.argmax(power)
    period = 1.0 / freq[argmax]
    false_alarm_prob = ls.false_alarm_probability(power[argmax])
    return {"period": period, "false_alarm_prob": false_alarm_prob}

catalog_w_features = catalog_w_features.map_rows(
    extract_period,
    # Column names specifying function arguments
    columns=["lc.hmjd", "lc.mag", "lc.magerr"],
    # Returned data type shape
    meta={"period": float, "false_alarm_prob": float},
    append_columns=True,
)
catalog_w_features

,objectid,filterid,objra,objdec,nepochs,lc,median_mag,n_lc,period,false_alarm_prob
npartitions=11,,,,,,,,,,
"Order: 5, Pixel: 2333",int64[pyarrow],int8[pyarrow],float[pyarrow],float[pyarrow],int64[pyarrow],"nested<hmjd: [double], mag: [float], magerr: [...",float64,int32,float64,float64
"Order: 5, Pixel: 2334",...,...,...,...,...,...,...,...,...,...
...,...,...,...,...,...,...,...,...,...,...
"Order: 5, Pixel: 2400",...,...,...,...,...,...,...,...,...,...
"Order: 5, Pixel: 2401",...,...,...,...,...,...,...,...,...,...


In [8]:
periodic_catalog = catalog_w_features.query("false_alarm_prob < 1e-10 and 1.5 < period < 9.5")

### Compute with Dask

In [9]:
%%time
from dask.distributed import Client
with Client(n_workers=4, memory_limit="auto"):
    dask_df = periodic_catalog.compute()

/astro/users/smcampos/.conda/envs/lsdb/lib/python3.13/site-packages/distributed/node.py:188: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 38037 instead
  warnings.warn(
/astro/users/smcampos/.conda/envs/lsdb/lib/python3.13/site-packages/numpy/_core/fromnumeric.py:3824: RuntimeWarning: Mean of empty slice
  return _methods._mean(a, axis=axis, dtype=dtype,
/astro/users/smcampos/.conda/envs/lsdb/lib/python3.13/site-packages/numpy/_core/_methods.py:142: RuntimeWarning: invalid value encountered in divide
  ret = ret.dtype.type(ret / rcount)
/astro/users/smcampos/.conda/envs/lsdb/lib/python3.13/site-packages/numpy/_core/fromnumeric.py:3824: RuntimeWarning: Mean of empty slice
  return _methods._mean(a, axis=axis, dtype=dtype,
/astro/users/smcampos/.conda/envs/lsdb/lib/python3.13/site-packages/numpy/_core/_methods.py:142: RuntimeWarning: invalid value encountered in divide
  ret = ret.dtype.type(ret / rcount)
/astro/use

CPU times: user 1.7 s, sys: 383 ms, total: 2.08 s
Wall time: 1min 13s


### Compute with ray

To install ray execute `%pip install -U ray[default] ipywidgets`.

In [10]:
import ray
from ray.util.dask import enable_dask_on_ray

In [11]:
%%time
ray.init(num_cpus=4, include_dashboard=True, dashboard_port=8265)
with enable_dask_on_ray():
    ray_df = periodic_catalog.compute()
ray.shutdown()

2026-04-03 09:59:02,368	INFO worker.py:2004 -- Started a local Ray instance. View the dashboard at http://127.0.0.1:8265 
/astro/users/smcampos/.conda/envs/lsdb/lib/python3.13/site-packages/ray/_private/worker.py:2052: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(
/astro/users/smcampos/.conda/envs/lsdb/lib/python3.13/site-packages/dask/config.py:835: FutureWarning: Dask configuration key 'shuffle' has been deprecated; please use 'dataframe.shuffle.algorithm' instead
  warnings.warn(


Computing Catalog:   0%|          | 0/177 [00:00<?, ?it/s]

(dask:('lambda-879d6fcc6d77488bc072a62a3676ecf5', 10) pid=3921362) /astro/users/smcampos/.conda/envs/lsdb/lib/python3.13/site-packages/numpy/_core/fromnumeric.py:3824: RuntimeWarning: Mean of empty slice
(dask:('lambda-879d6fcc6d77488bc072a62a3676ecf5', 10) pid=3921362)   return _methods._mean(a, axis=axis, dtype=dtype,
(dask:('lambda-879d6fcc6d77488bc072a62a3676ecf5', 10) pid=3921362) /astro/users/smcampos/.conda/envs/lsdb/lib/python3.13/site-packages/numpy/_core/_methods.py:142: RuntimeWarning: invalid value encountered in divide
(dask:('lambda-879d6fcc6d77488bc072a62a3676ecf5', 10) pid=3921362)   ret = ret.dtype.type(ret / rcount)
(dask:('lambda-879d6fcc6d77488bc072a62a3676ecf5', 1) pid=3921362) /astro/users/smcampos/.conda/envs/lsdb/lib/python3.13/site-packages/numpy/_core/fromnumeric.py:3824: RuntimeWarning: Mean of empty slice
(dask:('lambda-879d6fcc6d77488bc072a62a3676ecf5', 1) pid=3921362)   return _methods._mean(a, axis=axis, dtype=dtype,
(dask:('lambda-879d6fcc6d77488bc072a62

CPU times: user 1.02 s, sys: 341 ms, total: 1.36 s
Wall time: 1min 39s


In [12]:
import pandas.testing as pdt
pdt.assert_frame_equal(dask_df, ray_df)

- Ray was significantly slower. 
- Need to see the ray dashboard for more info.
- Ray needs to convert the graph so that might be why.